In [ ]:
#import necessary libraries
import torch
import pandas as pd
import rasterio
import torch.nn as nn
import numpy as np

from torch.utils.data import Dataset, DataLoader
from basicUnet import BasicUnet

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = BasicUnet(
    n_channels=4, #UAVSAR has 4 channels: HH, HV, VH, VV
    n_classes=1
).to(device)

In [ ]:
#define dataset class which takes csv as input. Returns the image and mask tensors for the model
class FloodDataset(Dataset):

    def __init__(self, csv_file):
        self.df = pd.read_csv(csv_file)

    def __len__(self):
        return len(self.df)
    #idx = each row in csv file
    def __getitem__(self, idx):

        image_path = self.df.iloc[idx]["uavsar_path"] #UAVSAR image
        mask_path = self.df.iloc[idx]["flood_mask_path"] #Corresponding ground truth flood mask

        with rasterio.open(image_path) as src:
            image = src.read().astype(np.float32) #read image and convert to float32. reads all channels

        with rasterio.open(mask_path) as src:
            mask = src.read(1).astype(np.float32) #read mask and convert to float32. read(1) reads the first band of the mask

        image = torch.tensor(image) #convert to tensor
        mask = torch.tensor(mask).unsqueeze(0) #convert to tensor and define a channel dimension (1 channel)

        return image, mask

In [ ]:
#the following parameters are meant to reproduce the SPIE 2026 paper results.

In [ ]:

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
#DICE loss calculation
class DiceLoss(nn.Module):

    def __init__(self, smooth=1):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):

        pred = pred.view(-1)
        target = target.view(-1)

        intersection = (pred * target).sum()

        dice = (
            2.0 * intersection + self.smooth
        ) / (
            pred.sum() + target.sum() + self.smooth
        )

        return 1 - dice

In [ ]:
#combined dice loss and BCE as described in the SPIE 2026 paper
bce_loss = nn.BCELoss()
dice_loss = DiceLoss()

def combined_loss(pred, target):

    bce = bce_loss(pred, target)
    dice = dice_loss(pred, target)

    return bce + dice

In [ ]:
#load csv into dataset
train_dataset = FloodDataset("csv_splits/flood_splits_standard_strict_train_val/standard/heldout_fp1_train.csv")

#create dataloader for training
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

In [ ]:
#training loop

num_epochs = 80

for epoch in range(num_epochs):

    model.train()

    running_loss = 0

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device).float()

        optimizer.zero_grad()

        predictions = model(images)

        loss = combined_loss(
            predictions,
            masks
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}/80 "
        f"Loss: {running_loss/len(train_loader):.4f}"
    )